API Pipeline Notebook for Team 1000 Research Question: "Which weather factor has the biggest impact on traffic delays in New York?"

In [3]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime


We're using Open-Meteo because it has free historical weather history for a year and it says the type of weather.


In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

Setup the Open-Meteo API client with cache and retry on error

In [ ]:

cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


Getting the different parameters we need for weather Analysis

In [ ]:
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 40.7143,
	"longitude": -74.006,
	"start_date": "2024-01-01",
	"end_date": "2024-12-31",
	"hourly": ["temperature_2m", "snowfall", "showers", "rain", "snow_depth", "weather_code"],
	"timezone": "auto",
	"wind_speed_unit": "mph",
	"temperature_unit": "fahrenheit",
	"precipitation_unit": "inch",
}
responses = openmeteo.weather_api(url, params = params)


In [16]:
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E") #New York Location
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 40.71033477783203°N -73.99308013916016°E
Elevation: 51.0 m asl
Timezone: b'America/New_York'b'GMT-4'
Timezone difference to GMT+0: -14400s


In [19]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(1).ValuesAsNumpy()
hourly_showers = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(4).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["snowfall"] = hourly_snowfall
hourly_data["showers"] = hourly_showers
hourly_data["rain"] = hourly_rain
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["weather_code"] = hourly_weather_code

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Hourly data
                           date  temperature_2m  snowfall  showers      rain  \
0    2024-01-01 00:00:00+00:00       38.763500       0.0      0.0  0.000000   
1    2024-01-01 01:00:00+00:00       40.923500       0.0      0.0  0.000000   
2    2024-01-01 02:00:00+00:00       40.653500       0.0      0.0  0.000000   
3    2024-01-01 03:00:00+00:00       40.653500       0.0      0.0  0.000000   
4    2024-01-01 04:00:00+00:00       39.663502       0.0      0.0  0.000000   
...                        ...             ...       ...      ...       ...   
8779 2024-12-31 19:00:00+00:00       46.683498       0.0      0.0  0.000000   
8780 2024-12-31 20:00:00+00:00       47.403503       0.0      0.0  0.000000   
8781 2024-12-31 21:00:00+00:00       47.313499       0.0      0.0  0.000000   
8782 2024-12-31 22:00:00+00:00       47.043499       0.0      0.0  0.015748   
8783 2024-12-31 23:00:00+00:00       43.983498       0.0      0.0  0.015748   

      snow_depth  weather_code  
0   

Now we will obtain traffic Data from NYC Open Data

Now we must connect the Traffic and Weather Data that way we can observe trends